### take anatomically defined TPJ (angular gyrus, supramarginal gyrus and LOC superior in harvard_oxford atlas), and run a univariate ROI analysis on comparing the timeline of that anatomically defined TPJ to MAC moral-non-moral


#### === TPJ mask creation & hemisphere split ===
#### Combines Harvard–Oxford anatomical ROIs with a Neurosynth "TPJ" meta-analytic mask (neurosynth.org)
#### Then splits the refined mask into left/right hemispheres (MNI x<0 / x>0)


In [ ]:
#pip install nimare

In [6]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os, requests
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl
import os
import subprocess

alias_dir = "/Users/silvycollin/Documents/research/narratives/fmriprep/"

from nilearn import datasets, image, masking
import numpy as np

import nimare

try:
    # Newer NiMARE versions
    from nimare.datasets import fetch_neurosynth
except ImportError:
    # Older NiMARE versions
    from nimare.extract import fetch_neurosynth



In [11]:
# task = '21styear'
# print( "starting download of " + task )
# for root, dirs, files in os.walk(alias_dir):
#     for file in files:
#         if str(task) in file and file.endswith("space-MNI152NLin2009cAsym_res-native_desc-preproc_bold.nii.gz"):
#             epi = os.path.join(root,file)
#             print("Downloading data for %s" % (epi))
#             subprocess.call(["datalad","get", epi], shell=True)
#         if str(task) in file and file.endswith("desc-confounds_regressors.tsv"):
#             epi = os.path.join(root, file)
#             print("Downloading data for %s" % (epi))
#             subprocess.call(["datalad", "get", epi], shell=True)

In [2]:
fcontrast_dir = "/Users/silvycollin/Documents/research/narrativeProject/processed_first_level_per_sentence/virtues/fcontrast/21styear/"


In [4]:
# ----------------------------
# 1. Load the downloaded NeuroSynth TPJ map
# ----------------------------
ns_img = image.load_img("tpj_association-test_z_FDR_0.01.nii")

# Threshold (e.g., z >= 3) and binarize
ns_data = ns_img.get_fdata()
ns_bin = (ns_data >= 3.0).astype(int)
ns_bin_img = image.new_img_like(ns_img, ns_bin)
ns_bin_img.to_filename("TPJ_ns_mask_z3.nii.gz")
print("Saved thresholded NeuroSynth TPJ mask as TPJ_ns_mask_z3.nii.gz")
print("Nonzero voxels:", np.sum(ns_bin))

# ----------------------------
# 2. Load Harvard-Oxford cortical atlas
# ----------------------------
atlas = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr25-2mm')
atlas_img = image.load_img(atlas.maps)
atlas_data = atlas_img.get_fdata()
labels = atlas.labels

# Select anatomical TPJ regions
target_regions = [
    "Angular Gyrus",
    "Supramarginal Gyrus",
    "Lateral Occipital Cortex, superior division"
]

indices = [i for i, name in enumerate(labels) if any(r in name for r in target_regions)]
print("Selected HO indices:", indices)
print("Selected labels:", [labels[i] for i in indices])

# Create anatomical mask
anat_mask_data = np.isin(atlas_data, indices).astype(int)
anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
anat_mask_img.to_filename("TPJ_HO_mask.nii.gz")
print("Saved anatomical TPJ mask as TPJ_HO_mask.nii.gz")

# ----------------------------
# 3. Intersect anatomical & meta-analytic masks
# ----------------------------
refined_data = anat_mask_data * ns_bin
refined_img = image.new_img_like(atlas_img, refined_data)
refined_img.to_filename("TPJ_refined_mask.nii.gz")
print("Saved final refined TPJ mask as TPJ_refined_mask.nii.gz")
print("Nonzero voxels:", np.sum(refined_data))

# ----------------------------
# 4. Split into left / right hemispheres
# ----------------------------
affine = refined_img.affine
coords = np.indices(refined_data.shape)
# MNI x-coordinates
x_coords = affine[0, 0] * coords[0] + affine[0, 3]

left_mask = (refined_data > 0) & (x_coords < 0)
right_mask = (refined_data > 0) & (x_coords > 0)

left_img = image.new_img_like(refined_img, left_mask.astype(int))
right_img = image.new_img_like(refined_img, right_mask.astype(int))

left_img.to_filename("TPJ_refined_mask_L.nii.gz")
right_img.to_filename("TPJ_refined_mask_R.nii.gz")

print("Saved left hemisphere mask as TPJ_refined_mask_L.nii.gz")
print("Saved right hemisphere mask as TPJ_refined_mask_R.nii.gz")
print("Left voxels:", np.sum(left_mask), "Right voxels:", np.sum(right_mask))

Saved thresholded NeuroSynth TPJ mask as TPJ_ns_mask_z3.nii.gz
Nonzero voxels: 3014


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15261/4091735115.py:9: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  ns_bin_img = image.new_img_like(ns_img, ns_bin)


[get_dataset_dir] Dataset found in /Users/silvycollin/nilearn_data/fsl

Selected HO indices: [19, 20, 21, 22]
Selected labels: ['Supramarginal Gyrus, anterior division', 'Supramarginal Gyrus, posterior division', 'Angular Gyrus', 'Lateral Occipital Cortex, superior division']
Saved anatomical TPJ mask as TPJ_HO_mask.nii.gz
Saved final refined TPJ mask as TPJ_refined_mask.nii.gz
Nonzero voxels: 1719
Saved left hemisphere mask as TPJ_refined_mask_L.nii.gz
Saved right hemisphere mask as TPJ_refined_mask_R.nii.gz
Left voxels: 1079 Right voxels: 640


/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15261/4091735115.py:35: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  anat_mask_img = image.new_img_like(atlas_img, anat_mask_data)
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15261/4091735115.py:43: UserWarning: Data array used to create a new image contains 64-bit ints. This is likely due to creating the array with numpy and passing `int` as the `dtype`. Many tools such as FSL and SPM cannot deal with int64 in Nifti images, so for compatibility the data has been converted to int32.
  refined_img = image.new_img_like(atlas_img, refined_data)
/var/folders/2b/y7vqbp2s3slchmzyrsz69p1m0000gn/T/ipykernel_15261/4091735115.py:59: UserWarning: Data array used to create a new imag